# Neuropsychology Merge

Load neuropsychology Excel files, normalize participant IDs to `uid` and `timepoint`, then merge into one longitudinal table.

In [19]:
import pandas as pd
from pathlib import Path

# Input files
xlsx_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\memor_rappel_libre_indice_memoria.xlsx")
ref_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\2026-04-04_glu_moca_struc_id_age_t1_t2.csv")

# Output file (saved under neuropsy)
out_path = ref_path.parent / "memor_rappel_libre_indice_memoria_wide_t1_t2.csv"

# Load data
mem = pd.read_excel(xlsx_path)
ref = pd.read_csv(ref_path)

def _norm(s: str) -> str:
    return "".join(ch for ch in str(s).lower().strip() if ch.isalnum())

def _find_col(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    norm_map = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    raise KeyError(f"Could not find {label}. Available columns: {list(df.columns)}")

# Flexible column detection to avoid exact-name failures
mem_id_col = _find_col(mem, ["psid", "participantid", "id", "uid"], "memoria ID column")
mem_age_col = _find_col(mem, ["age", "aget1", "agevisit", "ageyears"], "memoria age column")

ref_id_col = _find_col(ref, ["psid", "participantid", "id", "uid"], "reference ID column")
ref_age_t1_col = _find_col(ref, ["aget1", "age_t1", "agetp1"], "reference age_t1 column")
ref_age_t2_col = _find_col(ref, ["aget2", "age_t2", "agetp2"], "reference age_t2 column")

# Value columns from memoria file (exclude keys and technical fields)
exclude_norm = {_norm(mem_id_col), _norm(mem_age_col), "timepoint", "tp", "visit", "session"}
value_cols = [c for c in mem.columns if _norm(c) not in exclude_norm]
if not value_cols:
    raise ValueError("No value columns found in memoria file after excluding ID, age, and technical columns.")

# Normalize types for matching
mem_long = mem.copy()
mem_long[mem_id_col] = mem_long[mem_id_col].astype(str).str.strip()
mem_long[mem_age_col] = pd.to_numeric(mem_long[mem_age_col], errors="coerce")

ref_wide = ref.copy()
ref_wide[ref_id_col] = ref_wide[ref_id_col].astype(str).str.strip()
ref_wide[ref_age_t1_col] = pd.to_numeric(ref_wide[ref_age_t1_col], errors="coerce")
ref_wide[ref_age_t2_col] = pd.to_numeric(ref_wide[ref_age_t2_col], errors="coerce")

# Build long reference with timepoint labels from age_t1/age_t2
ref_long_t1 = ref_wide[[ref_id_col, ref_age_t1_col]].rename(columns={ref_age_t1_col: "age"})
ref_long_t1["timepoint"] = "t1"

ref_long_t2 = ref_wide[[ref_id_col, ref_age_t2_col]].rename(columns={ref_age_t2_col: "age"})
ref_long_t2["timepoint"] = "t2"

ref_long = pd.concat([ref_long_t1, ref_long_t2], ignore_index=True)
ref_long = ref_long.rename(columns={ref_id_col: "PSID"})
ref_long["PSID"] = ref_long["PSID"].astype(str).str.strip()

# Prepare memoria long on standardized keys
mem_std = mem_long.rename(columns={mem_id_col: "PSID", mem_age_col: "age"})

# Match by PSID + age, keep timepoint from reference
matched_long = ref_long.merge(mem_std, on=["PSID", "age"], how="left")

# Use the reference timepoint column even if merge suffixes were added
if "timepoint" in matched_long.columns:
    tp_col = "timepoint"
elif "timepoint_x" in matched_long.columns:
    tp_col = "timepoint_x"
else:
    raise KeyError("Could not find timepoint column after merge.")

# Pivot to wide so each measure gets _t1/_t2 columns
wide = matched_long.pivot_table(
    index="PSID",
    columns=tp_col,
    values=value_cols,
    aggfunc="first"
)

# Flatten multi-index columns: measure_t1 / measure_t2
wide.columns = [f"{measure}_{tp}" for measure, tp in wide.columns]
wide = wide.reset_index()

# Keep participant order from reference and include both ages
wide = ref_wide[[ref_id_col, ref_age_t1_col, ref_age_t2_col]].rename(columns={ref_id_col: "PSID"}).merge(wide, on="PSID", how="left")

# Save
wide.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows saved: {len(wide)}")
wide.head()

Saved: C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\memor_rappel_libre_indice_memoria_wide_t1_t2.csv
Rows saved: 75


,PSID,age_t1,age_t2,date_evaluation_t1,date_evaluation_t2,evaluation_t1,evaluation_t2,id_participant_t1,id_participant_t2,memor_memoria_score_total_15_t1,...,memor_rappel_libre_nombre_mots_doubles_sans_commentaire_t1,memor_rappel_libre_nombre_mots_doubles_sans_commentaire_t2,memor_rappel_libre_nombre_reponses_correctes_t1,memor_rappel_libre_nombre_reponses_correctes_t2,memor_score_indice_double_avec_com_t1,memor_score_indice_double_avec_com_t2,memor_score_indice_double_sans_com_t1,memor_score_indice_double_sans_com_t2,uid_t1,uid_t2
0,3002498,72.6,74.6,2019-05-13,2021-05-13,neuropsy,neuropsy,3002498.0,3002498.0,15.0,...,0,0,11.0,9.0,0,0,0,0,3002498_neuropsy_t04,3002498_neuropsy_t06
1,3025432,71.6,75.4,2018-08-13,2022-05-18,neuropsy,neuropsy,3025432.0,3025432.0,15.0,...,0,0,10.0,11.0,0,0,0,0,3025432_neuropsy_t02,3025432_neuropsy_t06
2,3100205,73.6,78.0,2019-12-02,2024-05-07,neuropsy,neuropsy,3100205.0,3100205.0,13.0,...,3,0,6.0,4.0,0,0,0,0,3100205_neuropsy_t04,3100205_neuropsy_t08
3,3123186,71.7,73.6,2019-04-01,2021-03-18,neuropsy,neuropsy,3123186.0,3123186.0,14.0,...,1,0,5.0,7.0,0,0,0,0,3123186_neuropsy_t02,3123186_neuropsy_t04
4,3149469,70.2,72.2,2022-04-14,NaT,neuropsy,NaN,3149469.0,NaN,15.0,...,0,NaN,12.0,NaN,0,NaN,0,NaN,3149469_neuropsy_t04,NaN


In [18]:
import pandas as pd
from pathlib import Path

# Input files
xlsx_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\prenom_visage.xlsx")
ref_path = Path(r"C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\2026-04-04_glu_moca_struc_id_age_t1_t2.csv")

# Output file (saved under neuropsy)
out_path = ref_path.parent / "prenom_visage_wide_t1_t2.csv"

# Load data
mem = pd.read_excel(xlsx_path)
ref = pd.read_csv(ref_path)

def _norm(s: str) -> str:
    return "".join(ch for ch in str(s).lower().strip() if ch.isalnum())

def _find_col(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    norm_map = {_norm(c): c for c in df.columns}
    for cand in candidates:
        if cand in norm_map:
            return norm_map[cand]
    raise KeyError(f"Could not find {label}. Available columns: {list(df.columns)}")

# Flexible column detection to avoid exact-name failures
mem_id_col = _find_col(mem, ["psid", "participantid", "id", "uid"], "prenom_visage ID column")
mem_age_col = _find_col(mem, ["age", "aget1", "agevisit", "ageyears"], "prenom_visage age column")

ref_id_col = _find_col(ref, ["psid", "participantid", "id", "uid"], "reference ID column")
ref_age_t1_col = _find_col(ref, ["aget1", "age_t1", "agetp1"], "reference age_t1 column")
ref_age_t2_col = _find_col(ref, ["aget2", "age_t2", "agetp2"], "reference age_t2 column")

# Value columns from prenom_visage file (exclude keys and technical fields)
exclude_norm = {_norm(mem_id_col), _norm(mem_age_col), "timepoint", "tp", "visit", "session"}
value_cols = [c for c in mem.columns if _norm(c) not in exclude_norm]
if not value_cols:
    raise ValueError("No value columns found in prenom_visage file after excluding ID, age, and technical columns.")

# Normalize types for matching
mem_long = mem.copy()
mem_long[mem_id_col] = mem_long[mem_id_col].astype(str).str.strip()
mem_long[mem_age_col] = pd.to_numeric(mem_long[mem_age_col], errors="coerce")

ref_wide = ref.copy()
ref_wide[ref_id_col] = ref_wide[ref_id_col].astype(str).str.strip()
ref_wide[ref_age_t1_col] = pd.to_numeric(ref_wide[ref_age_t1_col], errors="coerce")
ref_wide[ref_age_t2_col] = pd.to_numeric(ref_wide[ref_age_t2_col], errors="coerce")

# Build long reference with timepoint labels from age_t1/age_t2
ref_long_t1 = ref_wide[[ref_id_col, ref_age_t1_col]].rename(columns={ref_age_t1_col: "age"})
ref_long_t1["timepoint"] = "t1"

ref_long_t2 = ref_wide[[ref_id_col, ref_age_t2_col]].rename(columns={ref_age_t2_col: "age"})
ref_long_t2["timepoint"] = "t2"

ref_long = pd.concat([ref_long_t1, ref_long_t2], ignore_index=True)
ref_long = ref_long.rename(columns={ref_id_col: "PSID"})
ref_long["PSID"] = ref_long["PSID"].astype(str).str.strip()

# Prepare prenom_visage long on standardized keys
mem_std = mem_long.rename(columns={mem_id_col: "PSID", mem_age_col: "age"})

# Match by PSID + age, keep timepoint from reference
matched_long = ref_long.merge(mem_std, on=["PSID", "age"], how="left")

# Use the reference timepoint column even if merge suffixes were added
if "timepoint" in matched_long.columns:
    tp_col = "timepoint"
elif "timepoint_x" in matched_long.columns:
    tp_col = "timepoint_x"
else:
    raise KeyError("Could not find timepoint column after merge.")

# Pivot to wide so each measure gets _t1/_t2 columns
wide = matched_long.pivot_table(
    index="PSID",
    columns=tp_col,
    values=value_cols,
    aggfunc="first"
)

# Flatten multi-index columns: measure_t1 / measure_t2
wide.columns = [f"{measure}_{tp}" for measure, tp in wide.columns]
wide = wide.reset_index()

# Keep participant order from reference and include both ages
wide = ref_wide[[ref_id_col, ref_age_t1_col, ref_age_t2_col]].rename(columns={ref_id_col: "PSID"}).merge(wide, on="PSID", how="left")

# Save
wide.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows saved: {len(wide)}")
wide.head()

Saved: C:\Users\okkam\Desktop\labo\article 2\Longitudinal_Multimodal_Data_CIMAQ\neuropsy\prenom_visage_wide_t1_t2.csv
Rows saved: 75


,PSID,age_t1,age_t2,date_evaluation_t1,date_evaluation_t2,evaluation_t1,evaluation_t2,id_participant_t1,id_participant_t2,uid_t1,uid_t2,visage_visages_score_rappel_differe_9_t1,visage_visages_score_rappel_differe_9_t2,visage_visages_score_rappel_immediat_9_t1,visage_visages_score_rappel_immediat_9_t2,visage_visages_score_total_reconnaissances_9_t1,visage_visages_score_total_reconnaissances_9_t2
0,3002498,72.6,74.6,2019-05-13,2021-05-13,neuropsy,neuropsy,3002498.0,3002498.0,3002498_neuropsy_t04,3002498_neuropsy_t06,8.0,8.0,8.0,5.0,9.0,9.0
1,3025432,71.6,75.4,2018-08-13,2022-05-18,neuropsy,neuropsy,3025432.0,3025432.0,3025432_neuropsy_t02,3025432_neuropsy_t06,6.0,9.0,8.0,8.0,9.0,9.0
2,3100205,73.6,78.0,2019-12-02,2024-05-07,neuropsy,neuropsy,3100205.0,3100205.0,3100205_neuropsy_t04,3100205_neuropsy_t08,7.0,4.0,6.0,4.0,9.0,5.0
3,3123186,71.7,73.6,2019-04-01,2021-03-18,neuropsy,neuropsy,3123186.0,3123186.0,3123186_neuropsy_t02,3123186_neuropsy_t04,7.0,5.0,5.0,5.0,8.0,4.0
4,3149469,70.2,72.2,2022-04-14,NaT,neuropsy,NaN,3149469.0,NaN,3149469_neuropsy_t04,NaN,6.0,NaN,7.0,NaN,7.0,NaN
